# wmshare 选股实战：筛出「高股息 + PE 十年低位」

从全市场 5,000+ 家 A 股里找：**TTM 股息率 3%–10%、非 ST，PE 为正（0–40）且处于自身近十年低位（≤50% 分位）**。

逐票请求会撞限流（按请求次数计费，免费档 500 次/小时），所以先用**一次批量**拿全市场股息率（12 次请求），只对候选逐票查 PE 与十年分位——服务端预计算好的指标码直出，不用自己拉十年序列算分位。

In [1]:
import wmshare

cli = wmshare.Client.from_env()
pool = cli.stocks.list()                                     # 全市场代码池
meta = {r["symbol"]: r for r in pool}
yld = cli.batch_all("bonus.snapshot_yield_ttm", list(meta))  # 12 次请求，不是 5,691 次
print(f"全市场 {len(pool):,} 家 → {yld.requests} 次请求拿完股息率（错误 {len(yld.errors)} 个）")

全市场 5,691 家 → 12 次请求拿完股息率（错误 0 个）


In [2]:
import pandas as pd

top = sorted(((s, v) for s, v in yld.results.items() if isinstance(v, (int, float))),
             key=lambda kv: -kv[1])
cand = [(s, v) for s, v in top if 0.03 <= v <= 0.10 and not meta[s].get("is_st")][:60]
rows = []
for s, v in cand:
    m = cli.indicator.metrics(s, ["$PE_TTM_LAST", "$PCT_PE_TTM_10Y_LAST"])
    pe, pct = m["$PE_TTM_LAST"], m["$PCT_PE_TTM_10Y_LAST"]
    if pe and 0 < pe < 40 and pct is not None and pct <= 0.5:
        rows.append({"代码": s, "名称": meta[s]["name"],
                     "股息率": v, "PE": pe, "十年分位": pct})

hits = (pd.DataFrame(rows)
          .sort_values("十年分位")
          .assign(**{"股息率": lambda df: df["股息率"].map("{:.2%}".format),
                     "PE": lambda df: df["PE"].map("{:.1f}".format),
                     "十年分位": lambda df: df["十年分位"].map("{:.1%}".format)})
          .head(8)
          .set_index("代码"))
print(f"候选 {len(cand)} 只 → 命中 {len(rows)}，按十年分位取前 8")
hits

候选 60 只 → 命中 20，按十年分位取前 8


,名称,股息率,PE,十年分位
代码,,,,
920022,世昌股份,7.57%,14.2,0.0%
300910,瑞丰新材,7.18%,12.0,0.3%
002616,长青集团,8.57%,12.2,0.4%
000703,恒逸石化,7.49%,6.5,2.2%
002345,潮宏基,7.01%,9.7,3.0%
600983,惠而浦,8.78%,12.2,4.9%
002170,芭田股份,8.12%,10.9,5.6%
600351,亚宝药业,7.68%,13.7,11.9%


> 实测 2026-10-30 · 真实数据（凭证只走环境变量 / `~/.wmshare/credentials.env`，notebook 与输出不含任何密钥）。
> 数值随行情变动；复跑请自备开放平台凭证后按顺序执行上方单元。筛股结果仅供研究参考，不构成投资建议。